# 00 · Validated raw data and chronological partitions

This notebook is the only stage that reads the original IEEE-CIS files. It validates source files, performs safe transaction/identity joins, creates the chronological train/validation/holdout split **before any fitted preprocessing**, saves immutable prepared exports, and publishes a small pointer for notebooks 01–04.

Run with the project virtual environment. For a fast software check, set `FRAUD_NOTEBOOK_SMOKE=1` before starting the kernel. For genuine data, leave it unset. `FRAUD_NOTEBOOK_MAX_ROWS` can cap the labeled pool to the earliest chronological rows; `0` means no cap.


## 1. Environment and source scope


In [4]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository from either the project root or notebook/.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "src" / "fraud_detection").is_dir():
        PROJECT_ROOT = directory
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected src/fraud_detection."
    )

source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

SEED = 42
SMOKE = os.environ.get("FRAUD_NOTEBOOK_SMOKE", "0") == "1"
MAX_ROWS = int(os.environ.get("FRAUD_NOTEBOOK_MAX_ROWS", "0"))
if MAX_ROWS < 0:
    raise ValueError("FRAUD_NOTEBOOK_MAX_ROWS must be nonnegative.")

stage_setting = os.environ.get(
    "FRAUD_NOTEBOOK_STAGE_DIR", "data/processed/learning_stages"
)
STAGE_ROOT = Path(stage_setting).expanduser()
if not STAGE_ROOT.is_absolute():
    STAGE_ROOT = PROJECT_ROOT / STAGE_ROOT
STAGE_ROOT = STAGE_ROOT.resolve()
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "stage_root": str(STAGE_ROOT),
    "smoke": SMOKE,
    "max_rows": MAX_ROWS,
    "seed": SEED,
})

from fraud_detection.data import (
    calculate_file_sha256,
    chronological_split,
    dataframe_fingerprint,
)
from fraud_detection.data_preparation import (
    DataPreparation,
    load_prepared_data,
    validate_raw_manifest,
)
from fraud_detection.models import generated_fixture

raw_setting = os.environ.get("FRAUD_NOTEBOOK_RAW_DIR", "data/raw")
RAW_DIR = Path(raw_setting).expanduser()
if not RAW_DIR.is_absolute():
    RAW_DIR = PROJECT_ROOT / RAW_DIR
RAW_DIR = RAW_DIR.resolve()

SOURCE_MANIFEST = {}
SOURCE_FILE_SHA256 = {}
if SMOKE:
    DATA_SCOPE = "generated_fixture"
else:
    manifest_path = RAW_DIR / "data_manifest.json"
    if manifest_path.is_file():
        SOURCE_MANIFEST = validate_raw_manifest(RAW_DIR)
        DATA_SCOPE = SOURCE_MANIFEST["data_scope"]
    else:
        DATA_SCOPE = "real_full"
    if MAX_ROWS > 0:
        DATA_SCOPE = "real_sample"

print({"raw_dir": str(RAW_DIR) if not SMOKE else "not read", "data_scope": DATA_SCOPE})


{'project_root': 'C:\\Users\\Asus\\Downloads\\credit-card-fraud-detection-system-main\\credit-card-fraud-detection-system-main', 'stage_root': 'C:\\Users\\Asus\\Downloads\\credit-card-fraud-detection-system-main\\credit-card-fraud-detection-system-main\\data\\processed\\learning_stages', 'smoke': False, 'max_rows': 0, 'seed': 42}
{'raw_dir': 'C:\\Users\\Asus\\Downloads\\credit-card-fraud-detection-system-main\\credit-card-fraud-detection-system-main\\data\\raw', 'data_scope': 'real_full'}


## 2. Load and validate the four raw tables


In [3]:
preparation = DataPreparation(root=PROJECT_ROOT, raw_dir=RAW_DIR)

if SMOKE:
    # Decompose the generated joined fixture into transaction/identity tables so
    # the same merge code is exercised without pretending this is real evidence.
    fixture = generated_fixture(n_rows=240, seed=SEED)
    identity_columns = [
        c for c in fixture
        if c.startswith("id_") or c in ("DeviceInfo", "DeviceType")
    ]
    train_transaction = fixture.drop(columns=identity_columns)
    has_identity = fixture[identity_columns].notna().any(axis=1)
    train_identity = fixture.loc[
        has_identity, ["TransactionID", *identity_columns]
    ].copy()
    test_transaction = train_transaction.drop(columns="isFraud").copy()
    test_transaction["TransactionID"] += 1_000_000
    test_identity = train_identity.copy()
    test_identity["TransactionID"] += 1_000_000
else:
    train_transaction, train_identity, test_transaction, test_identity = preparation.load_data()
    for filename in (
        "train_transaction.csv",
        "train_identity.csv",
        "test_transaction.csv",
        "test_identity.csv",
    ):
        path = RAW_DIR / filename
        SOURCE_FILE_SHA256[filename] = calculate_file_sha256(path)

inventory = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": frame.shape[1]}
    for name, frame in {
        "train_transaction": train_transaction,
        "train_identity": train_identity,
        "test_transaction": test_transaction,
        "test_identity": test_identity,
    }.items()
])
display(inventory)

,table,rows,columns
0,train_transaction,590540,394
1,train_identity,144233,41
2,test_transaction,506691,393
3,test_identity,141907,41


## 3. Safe transaction + identity joins


In [5]:
labeled_pool, kaggle_test = preparation.merge_datasets(
    train_transaction, train_identity, test_transaction, test_identity
)

if SMOKE:
    # Restore fixture column order so equality checks are deterministic.
    labeled_pool = labeled_pool.loc[:, fixture.columns].copy()
    kaggle_test = kaggle_test.loc[:, fixture.drop(columns="isFraud").columns].copy()
    pd.testing.assert_frame_equal(labeled_pool, fixture, check_like=False)

if MAX_ROWS > 0:
    labeled_pool = (
        labeled_pool.sort_values(["TransactionDT", "TransactionID"], kind="stable")
        .head(MAX_ROWS)
        .copy()
    )
    DATA_SCOPE = "real_sample" if not SMOKE else DATA_SCOPE

labeled_pool.attrs["data_scope"] = DATA_SCOPE
kaggle_test.attrs["data_scope"] = DATA_SCOPE
preparation.quality_report["data_scope"] = DATA_SCOPE

display(pd.Series(preparation.quality_report["labeled_pool"], name="labeled_join_quality"))
display(pd.Series(preparation.quality_report["kaggle_test"], name="unlabeled_join_quality"))


schema_version                                                                         2
labeled                                                                             True
transaction_rows                                                                  590540
identity_rows                                                                     144233
merged_rows                                                                       590540
identity_matched_rows                                                             144233
identity_unmatched_transaction_rows                                               446307
identity_orphan_rows                                                                   0
identity_match_coverage                                                         0.244239
transaction_fingerprint                12c3a518eedb1ba16f0b6d825137404a9bdcb13a72f921...
identity_fingerprint                   78082c151056bbc7296a61ba48db88860608caac05d50f...
merged_fingerprint   

schema_version                                                                         2
labeled                                                                            False
transaction_rows                                                                  506691
identity_rows                                                                     141907
merged_rows                                                                       506691
identity_matched_rows                                                             141907
identity_unmatched_transaction_rows                                               364784
identity_orphan_rows                                                                   0
identity_match_coverage                                                         0.280066
transaction_fingerprint                b4ae9ed0537bc5904bb7a3412b0c2d612f9d66fa7001c3...
identity_fingerprint                   de99bd2c3d10b00a725e53bb364cdac113d5087dbe7a3f...
merged_fingerprint   

## 4. Chronological train / validation / holdout split


In [ ]:
partitions = chronological_split(
    labeled_pool,
    train_fraction=0.60,
    validation_fraction=0.20,
)
for name in ("train", "validation", "holdout"):
    getattr(partitions, name).attrs["data_scope"] = DATA_SCOPE

split_table = pd.DataFrame([
    {
        "partition": name,
        "rows": len(getattr(partitions, name)),
        "fraud_rate": float(getattr(partitions, name).isFraud.mean()),
        "TransactionDT_min": float(getattr(partitions, name).TransactionDT.min()),
        "TransactionDT_max": float(getattr(partitions, name).TransactionDT.max()),
    }
    for name in ("train", "validation", "holdout")
])
display(split_table)

assert partitions.train.TransactionDT.max() < partitions.validation.TransactionDT.min()
assert partitions.validation.TransactionDT.max() < partitions.holdout.TransactionDT.min()


## 5. Training-only exploratory audit


In [ ]:
training = partitions.train
predictors = training.drop(columns="isFraud")
missingness = predictors.isna().mean().sort_values(ascending=False)
summary = pd.Series({
    "training_rows": len(training),
    "predictor_columns": predictors.shape[1],
    "fraud_rows": int(training.isFraud.sum()),
    "fraud_rate": float(training.isFraud.mean()),
    "amount_median": float(training.TransactionAmt.median()),
    "amount_q95": float(training.TransactionAmt.quantile(0.95)),
    "columns_over_80pct_missing": int((missingness > 0.80).sum()),
})
display(summary)
display(missingness.head(20).rename("missing_fraction").to_frame())


## 6. Save immutable prepared exports and publish the handoff


In [ ]:
preparation.split_report = partitions.report
output_paths = preparation.save_data(
    labeled_pool,
    kaggle_test,
    partitions,
    data_scope=DATA_SCOPE,
)

# save_data publishes the manifest under the explicit `prepared_manifest` key.
prepared_manifest_path = Path(output_paths["prepared_manifest"]).resolve()
prepared_folder = prepared_manifest_path.parent

# Reload immediately. The updated loader verifies SHA-256 and recorded shape for
# every prepared Parquet file before returning it.
checked = load_prepared_data(prepared_folder)
for name in ("train", "validation", "holdout"):
    expected = getattr(partitions, name).reset_index(drop=True)
    restored = getattr(checked["partitions"], name).reset_index(drop=True)
    pd.testing.assert_frame_equal(expected, restored)

from fraud_detection.utils import frame_fingerprint
partition_fingerprints = {
    name: frame_fingerprint(getattr(partitions, name))
    for name in ("train", "validation", "holdout")
}

pointer = {
    "schema_version": 1,
    "prepared_folder": str(prepared_folder),
    "prepared_manifest": str(prepared_manifest_path),
    "prepared_manifest_sha256": calculate_file_sha256(prepared_manifest_path),
    "data_scope": DATA_SCOPE,
    "smoke": SMOKE,
    "max_rows": MAX_ROWS,
    "seed": SEED,
    "raw_dir": None if SMOKE else str(RAW_DIR),
    "source_file_sha256": SOURCE_FILE_SHA256,
    "partition_fingerprints": partition_fingerprints,
}
pointer_path = STAGE_ROOT / "prepared_latest.json"
pointer_path.write_text(json.dumps(pointer, indent=2), encoding="utf-8")

display(pd.Series(pointer, name="prepared_handoff"))
print("Next: run 01_feature_engineering.ipynb")


## What this notebook guarantees

- Raw data are never overwritten.
- Transaction/identity joins are validated.
- The split is chronological and timestamp groups do not cross boundaries.
- No imputation, encoding, scaling, feature selection or classifier is fitted here.
- Prepared files are checksum-verified when loaded by every later notebook.
